# Clase 4 — Chain-of-thought y prompts complejos

Hay tareas que requieren razonamiento en varios pasos: resolver un problema, tomar una decisión con múltiples criterios o analizar una situación con información incompleta. En esos casos, pedirle al modelo que piense en voz alta — **chain-of-thought** — mejora significativamente la precisión.

## Contenido

| Sección | Tema |
|---|---|
| 1 | Configuración del entorno |
| 2 | Qué es chain-of-thought y por qué funciona |
| 3 | CoT implícito vs. CoT explícito |
| 4 | CoT estructurado con pasos definidos |
| 5 | Restricciones y rúbricas dentro del prompt |
| 6 | Actividad: optimizar un prompt complejo |

---
## 1. Configuración del entorno

**Si es tu primera vez en este curso:**
1. Obtené tu API key en [aistudio.google.com](https://aistudio.google.com) → **Get API key**.
2. Guardala en `.env`:
   ```bash
   echo 'GEMINI_API_KEY=TU_CLAVE_AQUI' >> .env
   ```
3. Si no querés crear el archivo, la celda te la pide de forma interactiva.

In [ ]:
BACKEND = "local"

In [ ]:
import os
import getpass

GEMINI_MODEL = "gemini-2.5-flash-lite"

if BACKEND == "gemini":
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass
    GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")
    if not GEMINI_API_KEY:
        GEMINI_API_KEY = getpass.getpass("Ingresá tu API key de Gemini: ")

print(f"Backend: {BACKEND}")

In [ ]:
if BACKEND == "gemini":
    from google import genai
    from google.genai import types
    _cliente_gemini = genai.Client(api_key=GEMINI_API_KEY)
elif BACKEND == "local":
    from huggingface_hub import hf_hub_download
    from llama_cpp import Llama
    ruta_modelo = hf_hub_download(
        repo_id="unsloth/gemma-3-1b-it-GGUF",
        filename="gemma-3-1b-it-Q4_K_M.gguf"
    )
    _llm_local = Llama(model_path=ruta_modelo, n_ctx=2048, n_gpu_layers=0, verbose=False)


def llamar_llm(prompt, system_prompt="Sos un asistente útil y conciso.", temperature=0.7, max_tokens=300):
    if BACKEND == "gemini":
        r = _cliente_gemini.models.generate_content(
            model=GEMINI_MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=system_prompt,
                temperature=temperature,
                max_output_tokens=max_tokens,
            )
        )
        return r.text.strip()
    elif BACKEND == "local":
        r = _llm_local.create_chat_completion(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": prompt}
            ],
            temperature=temperature,
            max_tokens=max_tokens
        )
        return r["choices"][0]["message"]["content"].strip()


print(llamar_llm("Respondé solo: 'Entorno listo.'", max_tokens=10))

---
## 2. Qué es chain-of-thought y por qué funciona

Cuando un modelo responde directamente una pregunta compleja, puede saltarse pasos lógicos y llegar a una conclusión incorrecta. **Chain-of-thought (CoT)** es la técnica de pedirle que muestre su razonamiento paso a paso antes de dar la respuesta final.

Funciona porque:
- El modelo genera cada token en función de los anteriores. Si los pasos intermedios están escritos, el siguiente token es más probable que sea correcto.
- El razonamiento visible también te permite detectar dónde se equivocó.

| Tipo de tarea | ¿Vale la pena usar CoT? |
|---|---|
| Saludo, formato simple, resumen breve | No — el overhead no aporta |
| Razonamiento lógico o matemático | Sí — mejora mucho la precisión |
| Decisiones con múltiples criterios | Sí — fuerza a ponderar antes de concluir |
| Análisis de situaciones ambiguas | Sí — reduce conclusiones apresuradas |

---
## 3. CoT implícito vs. CoT explícito

Hay dos formas de activar el razonamiento paso a paso:

In [ ]:
# ─── Tarea de referencia ─────────────────────────────────────────────────────
# Una empresa tiene tres candidatos para un puesto. Necesita elegir uno
# considerando: experiencia, disponibilidad y presupuesto.

caso = """Una empresa necesita contratar un analista de datos.
Tiene tres candidatos:
- Ana: 5 años de experiencia, disponible en 2 semanas, pide $2500/mes.
- Bruno: 2 años de experiencia, disponible de inmediato, pide $1800/mes.
- Carmen: 8 años de experiencia, disponible en 2 meses, pide $3200/mes.
El proyecto empieza en 3 semanas y el presupuesto máximo es $2800/mes."""

pregunta = "¿A quién deberían contratar y por qué?"

# ─── Sin CoT: respuesta directa ──────────────────────────────────────────────
print("=== SIN CoT ===")
print(llamar_llm(f"{caso}\n\n{pregunta}", max_tokens=150))
print()

In [ ]:
# ─── CoT implícito: la frase mágica ──────────────────────────────────────────
# Solo agregar "Pensá paso a paso" activa el razonamiento en muchos casos.

print("=== CoT IMPLÍCITO ===")
prompt_cot_implicito = f"{caso}\n\n{pregunta}\n\nPensá paso a paso antes de responder."
print(llamar_llm(prompt_cot_implicito, max_tokens=300))
print()

In [ ]:
# ─── CoT explícito: pasos definidos por nosotros ─────────────────────────────
# Le indicamos exactamente qué analizar en cada paso.

print("=== CoT EXPLÍCITO ===")
prompt_cot_explicito = f"""{caso}

Para elegir al candidato, seguí estos pasos en orden:
Paso 1: Filtrá a los candidatos que superan el presupuesto máximo.
Paso 2: De los restantes, descartá a quienes no pueden empezar a tiempo.
Paso 3: De los que quedan, elegí el de mayor experiencia.
Paso 4: Escribí la recomendación final en una oración.

Muestra los pasos de pensamiento que realizas y la respuesta final."""

print(llamar_llm(prompt_cot_explicito, max_tokens=300))

> 💡 **Para discutir:** ¿Cuál de los tres enfoques fue más útil para este caso? ¿En qué tipo de decisiones de tu trabajo aplicarías CoT explícito?

---
## 4. CoT estructurado con pasos definidos

Cuando la tarea se repite muchas veces (por ejemplo, analizar contratos, evaluar propuestas, revisar reportes), conviene estandarizar los pasos del CoT en una plantilla reutilizable.

In [ ]:
# ─── Plantilla CoT para análisis de riesgo ───────────────────────────────────
# Caso de uso: evaluar si un proveedor nuevo es confiable.

def analizar_proveedor_cot(descripcion_proveedor):
    prompt = f"""Sos un analista de compras evaluando proveedores nuevos.

Información del proveedor:
{descripcion_proveedor}

Analizá siguiendo estos pasos:
Paso 1 — Experiencia: ¿Cuántos años opera y en qué industrias?
Paso 2 — Capacidad: ¿Puede cumplir el volumen y los plazos requeridos?
Paso 3 — Riesgos: Identificá hasta 2 riesgos concretos.
Paso 4 — Recomendación: Aprobado / Aprobado con condiciones / Rechazado + una oración de justificación.

Mostrá cada paso claramente numerado."""

    return llamar_llm(prompt, max_tokens=350)


proveedor_ejemplo = """TechSupply S.A. lleva 3 años en el mercado de insumos electrónicos.
Tiene capacidad para entregar 500 unidades mensuales con un plazo de 15 días.
No tiene certificaciones de calidad pero sí referencias de dos clientes medianos.
Su precio es un 20% menor al mercado actual."""

print(analizar_proveedor_cot(proveedor_ejemplo))

---
## 5. Restricciones y rúbricas dentro del prompt

Para casos donde la salida necesita cumplir criterios específicos, podemos incluir una **rúbrica** dentro del prompt: una lista explícita de condiciones que la respuesta debe satisfacer. El modelo la usa como checklist interno.

In [ ]:
# ─── Prompt con rúbrica explícita ─────────────────────────────────────────────
# Tarea: escribir un mensaje de feedback para un empleado.

situacion = """Lucas entregó el informe dos días tarde, pero la calidad del trabajo
fue muy buena. Es la primera vez que se retrasa en 18 meses de trabajo."""

prompt_con_rubrica = f"""Sos un líder de equipo. Escribí un mensaje de feedback para Lucas sobre esta situación:
{situacion}

El mensaje debe cumplir TODOS estos criterios:
✓ Reconocer primero lo positivo antes de mencionar el problema.
✓ Mencionar el retraso como un hecho, sin juicio sobre la persona.
✓ Proponer una acción concreta para evitar que se repita.
✓ Tono constructivo y directo, sin condescendencia.
✓ Máximo 5 oraciones en total.

Antes del mensaje, verificá en una línea que cumplís cada criterio."""

print(llamar_llm(prompt_con_rubrica, max_tokens=350))

In [ ]:
# ─── Comparación: sin rúbrica vs. con rúbrica ─────────────────────────────────

print("=== SIN RÚBRICA ===")
print(llamar_llm(
    f"Sos un líder de equipo. Escribí un mensaje de feedback para Lucas: {situacion}",
    max_tokens=200
))
print()
print("=== CON RÚBRICA ===")
print(llamar_llm(prompt_con_rubrica, max_tokens=350))

---
## 6. Actividad: optimizar un prompt complejo

Tomá una tarea de razonamiento o decisión de tu trabajo. Escribí primero un prompt simple y luego una versión con CoT explícito + rúbrica. Compará ambas respuestas.

In [ ]:
# TODO: Describí una situación que requiera razonamiento o decisión
mi_situacion = """..."""

# ─── Versión simple ───────────────────────────────────────────────────────────
# TODO: Escribí un prompt directo sin CoT
mi_prompt_simple = """..."""

print("=== VERSIÓN SIMPLE ===")
print(llamar_llm(mi_prompt_simple, max_tokens=200))
print()

In [ ]:
# TODO: Reescribí el mismo prompt con pasos CoT explícitos y una rúbrica (lista de pasos a seguir)
mi_prompt_cot = """...

Seguí estos pasos:
Paso 1: ...
Paso 2: ...
Paso 3: ...

La respuesta debe cumplir:
✓ ...
✓ ..."""

print("=== VERSIÓN CoT + RÚBRICA ===")
print(llamar_llm(mi_prompt_cot, max_tokens=350))